# Deep-SSFS — Notebook 1: Data Loading, Preprocessing & Classical Baselines


## 1. Environment Setup



In [ ]:
# Install pinned dependencies.
# Run once per runtime; restart the kernel afterwards if prompted.
!pip install -q numpy==1.26.4 pandas==2.1.4 scikit-learn-extra==0.3.0 xgboost --upgrade

In [ ]:
import os
import json
import datetime
import itertools
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.neighbors import NearestNeighbors, KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.datasets import fetch_olivetti_faces
from sklearn_extra.cluster import KMedoids
from scipy.linalg import eigh

from tensorflow.keras.datasets import mnist, fashion_mnist, cifar10



## 2. Global Configuration



In [ ]:
# ============================================================
# GLOBAL CONFIGURATION
# ============================================================

SEED = 42
RESULTS_DIR = './results/'
os.makedirs(RESULTS_DIR, exist_ok=True)

# None means use the full dataset (Olivetti Faces is always full).
DATASET_N_SAMPLES = {
    'Olivetti Faces': None,
    'MNIST':          2000,
    'Fashion-MNIST':  2000,
    'CIFAR-10':       2000,
}

# Feature-selection and graph parameters per dataset.
DATASET_PARAMS = {
    'Olivetti Faces': {'num_features': 30, 'num_clusters': 40, 'k': 6},
    'MNIST':          {'num_features': 50, 'num_clusters': 10, 'k': 5},
    'Fashion-MNIST':  {'num_features': 50, 'num_clusters': 10, 'k': 5},
    'CIFAR-10':       {'num_features': 30, 'num_clusters': 10, 'k': 6},
}

# Known Classic-SSFS 1-NN benchmark scores used as comparison baselines.
# None = benchmark not yet available for that configuration.
CLASSIC_BENCHMARK_SCORES = {
    'MNIST_2000':           0.6750,
    'MNIST_6000':           0.7450,
    'MNIST_10000':          0.6723,
    'Fashion-MNIST_2000':   0.6975,
    'Fashion-MNIST_6000':   0.7375,
    'Fashion-MNIST_10000':  0.7038,
    'CIFAR-10_1000':       None,
    'Olivetti_Faces_400':  None,
}

# Classic-SSFS baseline experiment settings.
BASELINE_DATASETS     = ['MNIST', 'Fashion-MNIST']
BASELINE_SAMPLE_SIZES = [2000, 4000, 6000, 10000]   # all sizes to benchmark
N_RESAMPLES           = 10    # inner SSFS resampling rounds
N_STABILITY_RUNS      = 3     # outer stability-selection repetitions
NUM_FULL_RUNS         = 3     # independent evaluation cycles for mean/std

# XGBoost hyperparameter grid (GPU-accelerated via hist + cuda).
XGB_GRID = {
    'n_estimators':     [200, 400],
    'max_depth':        [4, 6, 8],
    'learning_rate':    [0.05, 0.1],
    'subsample':        [0.8],
    'colsample_bytree': [0.8],
}
XGB_BASE = {
    'tree_method': 'hist',
    'device':      'cuda',
    'eval_metric': 'mlogloss',
    'verbosity':   0,
    'n_jobs':      -1,
}

np.random.seed(SEED)

def _ts():
    return datetime.datetime.now().strftime('%H:%M:%S')

print(f"[{_ts()}] Configuration loaded. Results will be saved to: {RESULTS_DIR}")

## 3. Dataset Loaders & Preprocessing



In [ ]:


def _subsample(X, y, n_samples):
    """Random subsample of rows without replacement."""
    if n_samples is not None and n_samples < X.shape[0]:
        idx = np.random.choice(X.shape[0], n_samples, replace=False)
        return X[idx], y[idx]
    return X, y


def load_olivetti_faces(n_samples=None):
    """Olivetti Faces dataset (400 images, 64×64 pixels, 40 subjects)."""
    print(f"[{_ts()}] Loading Olivetti Faces...")
    data = fetch_olivetti_faces(shuffle=True, random_state=SEED)
    return _subsample(data.data, data.target, n_samples)


def load_mnist_data(n_samples=2000):
    """MNIST handwritten digits (28×28 = 784 features)."""
    print(f"[{_ts()}] Loading MNIST (n={n_samples})...")
    (x_tr, y_tr), (x_te, y_te) = mnist.load_data()
    X = np.concatenate([x_tr, x_te]).reshape(-1, 784).astype('float32') / 255.0
    y = np.concatenate([y_tr, y_te])
    return _subsample(X, y, n_samples)


def load_fashion_mnist_data(n_samples=2000):
    """Fashion-MNIST clothing images (28×28 = 784 features)."""
    print(f"[{_ts()}] Loading Fashion-MNIST (n={n_samples})...")
    (x_tr, y_tr), (x_te, y_te) = fashion_mnist.load_data()
    X = np.concatenate([x_tr, x_te]).reshape(-1, 784).astype('float32') / 255.0
    y = np.concatenate([y_tr, y_te])
    return _subsample(X, y, n_samples)


def load_cifar10_data(n_samples=2000):
    """CIFAR-10 colour images (32×32×3 = 3072 features)."""
    print(f"[{_ts()}] Loading CIFAR-10 (n={n_samples})...")
    (x_tr, y_tr), (x_te, y_te) = cifar10.load_data()
    X = np.concatenate([x_tr, x_te]).reshape(-1, 3072).astype('float32') / 255.0
    y = np.concatenate([y_tr, y_te]).flatten()
    return _subsample(X, y, n_samples)


LOADER_MAP = {
    'Olivetti Faces': load_olivetti_faces,
    'MNIST':          load_mnist_data,
    'Fashion-MNIST':  load_fashion_mnist_data,
    'CIFAR-10':       load_cifar10_data,
}


def load_dataset(name, n_samples=None):
    """Unified loader — delegates to the appropriate per-dataset function."""
    loader = LOADER_MAP[name]
    return loader(n_samples) if n_samples is not None else loader()


def prepare_data(X, y, test_size=0.2, seed=SEED):
    """
    Stratified 80/20 train/test split followed by StandardScaler fitting
    exclusively on the training partition (no data leakage).

    Returns
    -------
    X_tr, X_te : scaled feature matrices
    y_tr, y_te : corresponding label arrays
    X_tr_raw, X_te_raw : unscaled versions (useful for pixel visualisation)
    """
    X_tr_raw, X_te_raw, y_tr, y_te = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=seed
    )
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr_raw)
    X_te = scaler.transform(X_te_raw)
    return X_tr, X_te, y_tr, y_te, X_tr_raw, X_te_raw


print("Data loaders and preprocessing utilities defined.")

## 4. Dataset Preview



In [ ]:
FASHION_LABELS = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]

def preview_dataset(X_raw, y, title, n_show=10, img_shape=(28, 28),
                    class_labels=None):
    """Plot one example per class from the raw (unnormalised) data."""
    fig, axes = plt.subplots(1, n_show, figsize=(n_show * 1.5, 2.2))
    fig.suptitle(title, fontsize=13, fontweight='bold')
    for i, ax in enumerate(axes):
        idx = np.where(y == i)[0][0]
        img = X_raw[idx].reshape(img_shape) if len(img_shape) == 2 else X_raw[idx].reshape(img_shape)
        ax.imshow(img, cmap='gray' if len(img_shape) == 2 else None)
        label = class_labels[i] if class_labels else str(i)
        ax.set_title(label, fontsize=8)
        ax.axis('off')
    plt.tight_layout()
    plt.show()


(X_mnist_raw, y_mnist_full), _ = mnist.load_data()
(X_fmnist_raw, y_fmnist_full), _ = fashion_mnist.load_data()

preview_dataset(
    X_mnist_raw.reshape(-1, 784) / 255.0, y_mnist_full,
    "MNIST — one example per class"
)
preview_dataset(
    X_fmnist_raw.reshape(-1, 784) / 255.0, y_fmnist_full,
    "Fashion-MNIST — one example per class",
    class_labels=FASHION_LABELS
)

## 5. Gaussian Noise Augmentation Utility



In [ ]:
def add_gaussian_noise(X, mean=0.0, std=0.2):
    noise = np.random.normal(mean, std, X.shape)
    return np.clip(X + noise, 0.0, 1.0)


# Demonstrate noise effect on one MNIST image.
sample = X_mnist_raw[0].reshape(1, 784) / 255.0
noisy  = add_gaussian_noise(sample, std=0.3)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(4, 2))
ax1.imshow(sample.reshape(28, 28), cmap='gray')
ax1.set_title('Original')
ax1.axis('off')
ax2.imshow(noisy.reshape(28, 28), cmap='gray')
ax2.set_title('Noisy (std=0.3)')
ax2.axis('off')
plt.tight_layout()
plt.show()

## 6. Evaluation Utility


In [ ]:
def evaluation(X_train, y_train, X_test, y_test):
    knn = KNeighborsClassifier(n_neighbors=1)
    knn.fit(X_train, y_train)
    return {
        'train': float(accuracy_score(y_train, knn.predict(X_train))),
        'test':  float(accuracy_score(y_test,  knn.predict(X_test))),
    }


def save_results(df, filename, extra_json=None, json_filename=None,
                 save_dir=RESULTS_DIR):

    ts   = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    stem = filename.rsplit('.', 1)[0]

    csv_path = os.path.join(save_dir, f"{stem}_{ts}.csv")
    try:
        df.to_csv(csv_path, index=False)
        print(f"  -> CSV  : {csv_path}")
    except Exception as e:
        print(f"  -> CSV save failed: {e}")

    if extra_json is not None and json_filename is not None:
        json_stem = json_filename.rsplit('.', 1)[0]
        json_path = os.path.join(save_dir, f"{json_stem}_{ts}.json")
        try:
            with open(json_path, 'w') as fh:
                json.dump(extra_json, fh, indent=4)
            print(f"  -> JSON : {json_path}")
        except Exception as e:
            print(f"  -> JSON save failed: {e}")


print("Evaluation and save utilities defined.")

## 7. Classical SSFS — XGBoost Surrogate

The original SSFS algorithm (from `classic-ssfs-benchmark.ipynb`) uses an XGBClassifier surrogate trained on pseudo-labels derived from spectral graph eigenvectors.


In [ ]:
class ClassicSSFS(BaseEstimator, TransformerMixin):
    """
    Classic Spectral Surrogate Feature Selection with an XGBoost surrogate.

    Parameters
    ----------
    n_features  : number of features to select
    n_clusters  : number of spectral pseudo-label clusters (= expected classes)
    k           : number of nearest neighbours for affinity matrix construction
    n_resamples : number of 80%-subsample passes to average importances over
    xgb_params  : dict of XGBClassifier hyperparameters (merged with XGB_BASE)
    """

    def __init__(self, n_features=50, n_clusters=10, k=5,
                 n_resamples=10, xgb_params=None, verbose=False):
        self.n_features  = n_features
        self.n_clusters  = n_clusters
        self.k           = k
        self.n_resamples = n_resamples
        self.xgb_params  = xgb_params if xgb_params is not None else {}
        self.verbose     = verbose
        self.selected_features_ = None
        self.final_scores_      = None

    def _build_affinity_matrix(self, X):
        """Adaptive-bandwidth Gaussian k-NN affinity matrix."""
        n    = X.shape[0]
        nbrs = NearestNeighbors(n_neighbors=self.k + 1, algorithm='auto').fit(X)
        distances, indices = nbrs.kneighbors(X)
        sigmas = np.maximum(distances[:, -1], 1e-8)
        W = np.zeros((n, n), dtype=np.float32)
        for i in range(n):
            for j in indices[i, 1:]:
                d_sq     = np.sum((X[i] - X[j]) ** 2)
                w        = np.exp(-d_sq / (sigmas[i] * sigmas[j]))
                W[i, j]  = W[j, i] = w
        return W

    def _get_xgb_model(self):
        params = {**XGB_BASE, **self.xgb_params}
        return xgb.XGBClassifier(**params)

    def fit(self, X, y=None):
        n_samples, n_orig = X.shape
        feature_scores = np.zeros(n_orig, dtype=np.float64)

        W          = self._build_affinity_matrix(X)
        D          = W.sum(axis=1)
        D_inv_sqrt = np.where(D > 0, D ** -0.5, 0.0)
        D_mat      = np.diag(D_inv_sqrt)
        L          = np.eye(n_samples, dtype=np.float32) - D_mat @ W @ D_mat

        n_eigs = min(self.n_clusters * 2, n_samples - 1)
        _, eigvecs = eigh(L, subset_by_index=[1, n_eigs])
        norms      = np.linalg.norm(eigvecs, axis=1, keepdims=True)
        eigvecs    = eigvecs / np.maximum(norms, 1e-8)

        for i in range(self.n_resamples):
            sub_size = int(0.8 * n_samples)
            idx      = np.random.choice(n_samples, sub_size, replace=False)
            X_s, ev_s = X[idx], eigvecs[idx]

            y_pseudo = KMedoids(
                n_clusters=self.n_clusters, random_state=i, method='alternate'
            ).fit(ev_s).labels_

            model = self._get_xgb_model()
            model.fit(X_s, y_pseudo)
            feature_scores += model.feature_importances_

        self.final_scores_      = feature_scores / self.n_resamples
        self.selected_features_ = np.argsort(self.final_scores_)[::-1][:self.n_features]
        return self

    def transform(self, X):
        if self.selected_features_ is None:
            raise RuntimeError("Call fit() before transform().")
        return X[:, self.selected_features_]


print("ClassicSSFS (XGBoost surrogate) defined.")

## 8. XGBoost Hyperparameter Grid Search



In [ ]:
def grid_search_xgb(X_train_pool, y_train_pool, dataset_params, val_size=0.2):
    """
    Exhaustive grid search over XGB_GRID.

    For each configuration:
      1. Internal train/val split of X_train_pool.
      2. Single-resample SSFS pass to select features.
      3. 1-NN accuracy on the val set as the objective.

    Returns the parameter dict with the highest 1-NN val accuracy.
    """
    all_combos = list(itertools.product(*XGB_GRID.values()))
    param_keys = list(XGB_GRID.keys())
    grid_size  = len(all_combos)

    print(f"[{_ts()}] Grid search: {grid_size} configurations ...")

    X_gs_tr, X_gs_val, y_gs_tr, y_gs_val = train_test_split(
        X_train_pool, y_train_pool,
        test_size=val_size, stratify=y_train_pool, random_state=0
    )

    best_acc, best_params = -1.0, None

    for combo_idx, combo in enumerate(all_combos, 1):
        params = dict(zip(param_keys, combo))
        try:
            ssfs = ClassicSSFS(
                n_features=dataset_params['num_features'],
                n_clusters=dataset_params['num_clusters'],
                k=dataset_params['k'],
                n_resamples=1,
                xgb_params=params,
            )
            ssfs.fit(X_gs_tr)
            sel     = ssfs.selected_features_
            knn     = KNeighborsClassifier(n_neighbors=1)
            knn.fit(X_gs_tr[:, sel], y_gs_tr)
            val_acc = accuracy_score(y_gs_val, knn.predict(X_gs_val[:, sel]))
        except Exception as exc:
            print(f"  [GS {combo_idx}/{grid_size}] FAILED ({exc})")
            continue

        marker = " <-- best" if val_acc > best_acc else ""
        print(f"  [GS {combo_idx:>2}/{grid_size}]  "
              f"n_est={params['n_estimators']:>3}  depth={params['max_depth']}  "
              f"lr={params['learning_rate']:.3f}  val_1NN={val_acc:.4f}{marker}")

        if val_acc > best_acc:
            best_acc, best_params = val_acc, params

    if best_params is None:
        best_params = {'n_estimators': 300, 'max_depth': 6,
                       'learning_rate': 0.1, 'subsample': 0.8,
                       'colsample_bytree': 0.8}
        print(f"[{_ts()}] All combos failed — using default parameters.")
    else:
        print(f"[{_ts()}] Best val 1-NN: {best_acc:.4f} | Params: {best_params}")

    return best_params




## 9. Classic-SSFS Stability Selection



In [ ]:
def run_classic_stability_selection(X_train, dataset_params, best_xgb_params,
                                     n_stability_runs, n_resamples):
    n_feats    = X_train.shape[1]
    cumulative = np.zeros(n_feats, dtype=np.float64)
    valid_runs = 0
    last_ssfs  = None

    for i in range(n_stability_runs):
        try:
            m = ClassicSSFS(
                n_features=dataset_params['num_features'],
                n_clusters=dataset_params['num_clusters'],
                k=dataset_params['k'],
                n_resamples=n_resamples,
                xgb_params=best_xgb_params,
            )
            m.fit(X_train)
            if m.final_scores_ is not None:
                cumulative += m.final_scores_
                valid_runs += 1
                last_ssfs   = m
                print(f"  Stability run {i+1}/{n_stability_runs} done.")
        except Exception as exc:
            print(f"  [WARNING] Stability run {i+1} failed: {exc}")

    if valid_runs == 0:
        raise RuntimeError("All stability runs failed — check configuration.")

    avg_scores = cumulative / valid_runs
    selected   = np.argsort(avg_scores)[::-1][:dataset_params['num_features']]
    return avg_scores, selected, last_ssfs




## 10. Run Classic-SSFS Baseline Experiment
.

In [ ]:
print(f"[{_ts()}] CLASSIC-SSFS BASELINE — TUNED XGBoost")

rows_baseline     = []
best_xgb_per_run  = {}

for name in BASELINE_DATASETS:
    for n_samples in BASELINE_SAMPLE_SIZES:
        try:
            print(f"\n{'='*60}")
            print(f"[{_ts()}] Dataset: {name}  |  Samples: {n_samples}")
            print(f"{'='*60}")

            params = DATASET_PARAMS[name]

            X, y = load_dataset(name, n_samples)
            X_train, X_test, y_train, y_test, _, _ = prepare_data(X, y)
            print(f"[{_ts()}] Train: {X_train.shape}  |  Test: {X_test.shape}")

            best_xgb = grid_search_xgb(X_train, y_train, params)

            acc_train_list, acc_test_list = [], []
            best_selected, best_acc, best_ssfs_obj = None, 0.0, None

            for run_idx in range(NUM_FULL_RUNS):
                print(f"\n  --- Cycle {run_idx+1}/{NUM_FULL_RUNS} ---")
                avg_scores, selected, last_ssfs = run_classic_stability_selection(
                    X_train, params, best_xgb,
                    n_stability_runs=N_STABILITY_RUNS,
                    n_resamples=N_RESAMPLES,
                )
                scores = evaluation(
                    X_train[:, selected], y_train,
                    X_test[:, selected],  y_test,
                )
                acc_train_list.append(scores['train'])
                acc_test_list.append(scores['test'])
                print(f"  Train: {scores['train']:.4f}  |  Test (1-NN): {scores['test']:.4f}")

                if scores['test'] > best_acc:
                    best_acc, best_selected, best_ssfs_obj = scores['test'], selected, last_ssfs

            mean_train = float(np.mean(acc_train_list))
            mean_test  = float(np.mean(acc_test_list))
            std_test   = float(np.std(acc_test_list))

            bench_key = f"{name}_{n_samples}"
            old_score = CLASSIC_BENCHMARK_SCORES.get(bench_key)
            diff      = round(mean_test - old_score, 4) if old_score is not None else None

            print(f"\n[{_ts()}] RESULT | Mean Train: {mean_train:.4f}  "
                  f"Mean Test: {mean_test:.4f}  STD: {std_test:.4f}  "
                  f"Prior Benchmark: {old_score}  Diff: {diff}")

            ts_now    = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
            safe_name = name.replace(' ', '_').replace('-', '_')
            base      = f"ClassicSSFS_XGB_{safe_name}_{n_samples}samples"

            # Save feature scores as .npy for downstream analysis.
            if best_ssfs_obj is not None and best_ssfs_obj.final_scores_ is not None:
                np.save(
                    os.path.join(RESULTS_DIR, f"{base}_{ts_now}_scores.npy"),
                    best_ssfs_obj.final_scores_
                )

            selected_list = best_selected.tolist() if best_selected is not None else []
            json_payload  = {
                bench_key: {
                    'dataset':              name,
                    'n_samples':            n_samples,
                    'surrogate':            'XGBClassifier',
                    'best_xgb_params':      best_xgb,
                    'selected_features':    selected_list,
                    'mean_train_acc':       round(mean_train, 4),
                    'mean_test_acc':        round(mean_test, 4),
                    'std_test':             round(std_test, 4),
                }
            }
            best_xgb_per_run[bench_key] = json_payload[bench_key]

            row = {
                'Dataset':           name,
                'Samples':           n_samples,
                'Num Features':      params['num_features'],
                'Mean Train':        round(mean_train, 4),
                'Mean Test (1-NN)':  round(mean_test, 4),
                'STD Test':          round(std_test, 4),
                'Prior Benchmark':   old_score,
                'Diff':              diff,
                'Best n_estimators': best_xgb.get('n_estimators'),
                'Best max_depth':    best_xgb.get('max_depth'),
                'Best lr':           best_xgb.get('learning_rate'),
            }
            rows_baseline.append(row)

            save_results(
                pd.DataFrame([row]),
                filename=f'{base}_{ts_now}.csv',
                extra_json=json_payload,
                json_filename=f'{base}_{ts_now}.json',
            )

        except Exception:
            import traceback
            traceback.print_exc()
            print(f"  ERROR on {name} ({n_samples}) — see traceback above.")


print("CLASSIC-SSFS BASELINE  |  CONSOLIDATED RESULTS")
df_baseline = pd.DataFrame(rows_baseline)
if not df_baseline.empty:
    print(df_baseline[['Dataset', 'Samples', 'Mean Test (1-NN)', 'STD Test',
                        'Prior Benchmark', 'Diff']].to_string(index=False))
    ts_final = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    save_results(
        df_baseline,
        filename=f'ClassicSSFS_Baseline_ALL_{ts_final}.csv',
        extra_json=best_xgb_per_run,
        json_filename=f'ClassicSSFS_Baseline_ALL_{ts_final}.json',
    )
else:
    print("No results.")